# Deepfake Detection Training Pipeline
**Target Hardware**: 2x NVIDIA T4 GPUs (`DataParallel`, `fp16` mixed precision)

### Specifications
- **Class Balance**: 300 Real vs 300 Fake videos (50 per category across 6 fake types)
- **Partitioning**: GroupKFold by Video-ID (0% identity leakage)
- **Augmentations**: Light geometric and color augmentations (no blurring or lossy compression)
- **Model**: ConvNeXt-Small (768-d) + 2D FFT Log-Spectrum (128-d) fusion
- **Training**: 2-phase fine-tuning (3 epochs head warmup, 5 epochs differential LR)
- **Export**: PyTorch `.pth` checkpoint and ONNX opset 14 (`.onnx`)

In [ ]:
# Install dependencies and verify environment
!pip install timm facenet-pytorch albumentations grad-cam onnx onnxruntime pyyaml -q

import os, sys, re, random, time, cv2, torch, shutil, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.fft
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import timm
from facenet_pytorch import MTCNN
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 1. Setup & Configuration

In [ ]:
# Load configuration and set random seeds
try:
    from src.config import load_config
    CFG = load_config()
except ImportError:
    CFG = {
        'paths': {'kaggle_input': '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23', 'output_dir': '/kaggle/working/frames_v2'},
        'preprocessing': {'img_size': 224, 'padding_scale': 1.30, 'max_real_videos': 300, 'max_fake_per_dir': 150, 'frames_per_video': 15},
        'training': {'batch_size': 64, 'epochs_phase1': 3, 'epochs_phase2': 15, 'lr_phase1': 1e-3, 'lr_backbone': 1e-5, 'lr_head': 1e-4, 'seed': 42},
        'manipulation_types': {'all': ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures', 'FaceShifter'], 'held_out_loto': 'FaceShifter'}
    }

SEED = CFG.get('training', {}).get('seed', 42)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

BASE = CFG.get('paths', {}).get('kaggle_input', '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23')
OUTPUT_DIR = "/kaggle/working/frames_v2"
IMG_SIZE = CFG.get('preprocessing', {}).get('img_size', 224)
PADDING_SCALE = CFG.get('preprocessing', {}).get('padding_scale', 1.30)
FAKE_DIRS = CFG.get('manipulation_types', {}).get('all', ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures', 'FaceShifter'])
HELD_OUT_TYPE = CFG.get('manipulation_types', {}).get('held_out_loto', 'FaceShifter')

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(f"{OUTPUT_DIR}/real", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/fake", exist_ok=True)
print("Configuration loaded.")

## 2. Face Extraction

In [ ]:
# Extract face crops using GPU MTCNN
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
mtcnn = MTCNN(keep_all=True, post_process=False, device=device, select_largest=True)

def extract_video(v_path, out_dir, folder_name, v_name, frames_per_video=15):
    if not os.path.exists(v_path): return 0
    cap = cv2.VideoCapture(v_path)
    if not cap.isOpened(): return 0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return 0
    actual = min(frames_per_video, total)
    step = max(total // actual, 1)
    target_frames = set(i * step for i in range(actual))
    frames_pil, frames_rgb = [], []
    curr_frame = 0
    max_target = max(target_frames)
    while cap.isOpened() and len(frames_rgb) < actual and curr_frame <= max_target:
        if curr_frame in target_frames:
            ret, frame = cap.read()
            if ret and frame is not None:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames_rgb.append(rgb)
                frames_pil.append(Image.fromarray(rgb))
        else:
            cap.grab()
        curr_frame += 1
    cap.release()
    if not frames_pil: return 0
    try:
        boxes_list, _ = mtcnn.detect(frames_pil)
        if boxes_list is None: boxes_list = [None] * len(frames_pil)
    except Exception:
        boxes_list = [None] * len(frames_pil)
    saved = 0
    for idx, (rgb, boxes) in enumerate(zip(frames_rgb, boxes_list)):
        if boxes is None or len(boxes) == 0: continue
        h, w, _ = rgb.shape
        best_box = max(boxes, key=lambda b: (b[2]-b[0])*(b[3]-b[1]))
        x1, y1, x2, y2 = best_box[:4]
        bw, bh = x2 - x1, y2 - y1
        if bw < 10 or bh < 10: continue
        cx, cy = (x1 + x2)/2.0, (y1 + y2)/2.0
        nbw, nbh = bw * PADDING_SCALE, bh * PADDING_SCALE
        nx1, ny1 = max(0, int(cx - nbw/2.0)), max(0, int(cy - nbh/2.0))
        nx2, ny2 = min(w, int(cx + nbw/2.0)), min(h, int(cy + nbh/2.0))
        face = rgb[ny1:ny2, nx1:nx2]
        if face.size == 0 or face.shape[0] < 10 or face.shape[1] < 10: continue
        face_resized = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
        fname = f"{folder_name}_{v_name}_f{idx}.png"
        cv2.imwrite(os.path.join(out_dir, fname), cv2.cvtColor(face_resized, cv2.COLOR_RGB2BGR))
        saved += 1
    return saved

if os.path.exists(BASE):
    df_path = os.path.join(BASE, "Deepfakes")
    fake_videos = sorted([f for f in os.listdir(df_path) if f.endswith('.mp4')])[:150]
    
    real_ids = set()
    for f in fake_videos:
        match = re.search(r'(\d+)_(\d+)', f)
        if match:
            real_ids.add(match.group(1))
            real_ids.add(match.group(2))
    
    real_path = os.path.join(BASE, "original")
    real_count = 0
    for rid in tqdm(sorted(list(real_ids)), desc="Extracting Real Identities"):
        v_path = os.path.join(real_path, f"{rid}.mp4")
        real_count += extract_video(v_path, f"{OUTPUT_DIR}/real", "original", rid)
    print(f"Extracted {real_count} real frames.")
    
    for fd in FAKE_DIRS:
        fd_path = os.path.join(BASE, fd)
        if not os.path.exists(fd_path): continue
        fake_count = 0
        for f in tqdm(fake_videos, desc=f"Extracting {fd}"):
            v_path = os.path.join(fd_path, f)
            v_name = os.path.splitext(f)[0]
            fake_count += extract_video(v_path, f"{OUTPUT_DIR}/fake", fd, v_name)
        print(f"Extracted {fake_count} frames for {fd}.")

## 3. Video-ID GroupKFold Partitioning

In [ ]:
# Partition data by video ID to prevent identity leakage
def extract_identities(fname):
    base = os.path.splitext(fname)[0]
    base_no_frame = re.sub(r'_f\d+$', '', base)
    match_pair = re.search(r'(\d+)_(\d+)', base_no_frame)
    if match_pair:
        return match_pair.group(1), match_pair.group(2)
    match_single = re.search(r'(\d+)', base_no_frame)
    if match_single:
        return match_single.group(1), match_single.group(1)
    return base_no_frame.split('_')[0], base_no_frame.split('_')[0]

def perform_graph_split(samples, test_size=0.15, val_size=0.15):
    import networkx as nx
    G = nx.Graph()
    for filepath, label in samples:
        id1, id2 = extract_identities(os.path.basename(filepath))
        G.add_edge(id1, id2)
        
    components = list(nx.connected_components(G))
    random.shuffle(components)
    
    n_total = len(components)
    n_test = max(1, int(n_total * test_size))
    n_val = max(1, int(n_total * val_size))
    
    test_comps = components[:n_test]
    val_comps = components[n_test:n_test + n_val]
    train_comps = components[n_test + n_val:]
    
    def get_samples_for_comps(comp_list):
        comp_nodes = set.union(*comp_list) if comp_list else set()
        res = []
        for filepath, label in samples:
            id1, id2 = extract_identities(os.path.basename(filepath))
            if id1 in comp_nodes:
                res.append((filepath, label))
        return res
        
    train_samples = get_samples_for_comps(train_comps)
    val_samples = get_samples_for_comps(val_comps)
    test_samples = get_samples_for_comps(test_comps)
    
    return train_samples, val_samples, test_samples

real_files = [(os.path.join(OUTPUT_DIR, "real", f), 0) for f in os.listdir(f"{OUTPUT_DIR}/real")] if os.path.exists(f"{OUTPUT_DIR}/real") else []
fake_files = [(os.path.join(OUTPUT_DIR, "fake", f), 1) for f in os.listdir(f"{OUTPUT_DIR}/fake")] if os.path.exists(f"{OUTPUT_DIR}/fake") else []
all_samples = real_files + fake_files

print(f"Total Real Files: {len(real_files)} | Total Fake Files: {len(fake_files)}")
train_samples, val_samples, test_samples = perform_graph_split(all_samples)
random.shuffle(train_samples)

print(f"Train samples: {len(train_samples)} | Val samples: {len(val_samples)} | Test samples: {len(test_samples)}")

## 4. PyTorch Dataset & Augmentations

In [ ]:
# PyTorch Dataset and DataLoader initialization
img_comp = A.JPEGCompression(quality_lower=50, quality_upper=90, p=0.4) if hasattr(A, 'JPEGCompression') else A.ImageCompression(quality_range=(50, 90), p=0.4)
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), rotate=(-15, 15), p=0.5),
    A.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10, p=0.5),
    A.Downscale(scale_min=0.7, scale_max=0.9, p=0.3),
    img_comp,
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class DeepfakeDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if bgr is not None:
            img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        else:
            with Image.open(path) as pil_img:
                img = np.array(pil_img.convert("RGB"))
        if self.transform:
            img = self.transform(image=img)['image']
        else:
            img = torch.from_numpy(img.transpose(2,0,1)).float() / 255.0
            img = (img - torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)) / torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
        return img, torch.tensor(label, dtype=torch.float32)

batch_sz = CFG.get('training', {}).get('batch_size', 64)
train_ds = DeepfakeDataset(train_samples, train_transform)
val_ds = DeepfakeDataset(val_samples, eval_transform)
test_ds = DeepfakeDataset(test_samples, eval_transform)

train_labels = [s[1] for s in train_samples]
class_counts = np.bincount(train_labels)
class_weights = 1. / class_counts
sample_weights = [class_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=batch_sz, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_sz, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_sz, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoaders ready | Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)} | Test Batches: {len(test_loader)}")

## 5. Dual-Stream Model Architecture

In [ ]:
# HybridDeepfakeDetector model definition
class FFTFrequencyExtractor(nn.Module):
    def __init__(self, out_features=128):
        super().__init__()
        self.rgb_to_gray = nn.Conv2d(3, 1, kernel_size=1, bias=False)
        with torch.no_grad():
            self.rgb_to_gray.weight.data = torch.tensor([[[[0.299]], [[0.587]], [[0.114]]]], dtype=torch.float32)
        self.rgb_to_gray.weight.requires_grad = False
        
        self.conv_net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        self.fc = nn.Linear(128, out_features)
        
    def forward(self, x):
        with torch.amp.autocast(device_type="cuda", enabled=False):
            x_fp32 = x.to(torch.float32)
            mean = torch.tensor([0.485, 0.456, 0.406], device=x.device, dtype=torch.float32).view(1, 3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225], device=x.device, dtype=torch.float32).view(1, 3, 1, 1)
            raw_x = (x_fp32 * std + mean).clamp(0.0, 1.0)
            
            gray = self.rgb_to_gray(raw_x)
            fft_2d = torch.fft.rfft2(gray)
            magnitude = torch.abs(fft_2d)
            log_spectrum = torch.log(magnitude + 1e-5)
            
            flat_spectrum = log_spectrum.flatten(1)
            min_val = flat_spectrum.min(dim=1, keepdim=True)[0].unsqueeze(-1).unsqueeze(-1)
            max_val = flat_spectrum.max(dim=1, keepdim=True)[0].unsqueeze(-1).unsqueeze(-1)
            norm_spectrum = (log_spectrum - min_val) / (max_val - min_val + 1e-5)
        norm_spectrum = norm_spectrum.to(x.dtype)
        
        feat = self.conv_net(norm_spectrum)
        return self.fc(feat)

class HybridDeepfakeDetector(nn.Module):
    def __init__(self, backbone_name="convnext_base", pretrained=True, use_fft_branch=True, dropout=0.3):
        super().__init__()
        self.use_fft_branch = use_fft_branch
        self.spatial_backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        spatial_in_features = self.spatial_backbone.num_features
        
        if self.use_fft_branch:
            self.freq_extractor = FFTFrequencyExtractor(out_features=128)
            self.gate_fc = nn.Linear(spatial_in_features + 128, 128)
            nn.init.constant_(self.gate_fc.bias, -2.0)
            fusion_dim = spatial_in_features + 128
        else:
            self.freq_extractor = None
            self.gate_fc = None
            fusion_dim = spatial_in_features
            
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )
        
    def forward(self, x):
        spatial_feat = self.spatial_backbone(x)
        if self.use_fft_branch and self.freq_extractor is not None and self.gate_fc is not None:
            freq_raw = self.freq_extractor(x)
            gate = torch.sigmoid(self.gate_fc(torch.cat([spatial_feat, freq_raw], dim=1)))
            freq_feat = freq_raw * gate
            fused = torch.cat([spatial_feat, freq_feat], dim=1)
        else:
            fused = spatial_feat
        logits = self.classifier(fused)
        return logits.squeeze(-1)

def build_model(use_fft=True):
    model = HybridDeepfakeDetector(backbone_name="convnext_base", pretrained=True, use_fft_branch=use_fft)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    return model.to(device)

test_m = build_model(use_fft=True)
print(f"Model initialized on {device} | Total Parameters: {sum(p.numel() for p in test_m.parameters()):,}")

## 6. Two-Phase Training & Evaluation

In [ ]:
# Training and evaluation functions
def train_two_phase(model, train_loader, val_loader):
    epochs_p1 = CFG.get('training', {}).get('epochs_phase1', 3)
    epochs_p2 = CFG.get('training', {}).get('epochs_phase2', 5)
    lr_p1 = CFG.get('training', {}).get('lr_phase1', 1e-3)
    lr_backbone = CFG.get('training', {}).get('lr_backbone', 1e-5)
    lr_head = CFG.get('training', {}).get('lr_head', 1e-4)
    criterion = nn.BCEWithLogitsLoss()
    best_val_auc = 0.0
    best_weights = None
    
    # Phase 1: Train classifier head (backbone frozen)
    print("Phase 1: Training classifier head (backbone frozen)")
    unwrapped = model.module if hasattr(model, 'module') else model
    for p in unwrapped.spatial_backbone.parameters():
        p.requires_grad = False
        
    head_params = [p for n, p in model.named_parameters() if "spatial_backbone" not in n and p.requires_grad]
    optimizer_p1 = torch.optim.AdamW(head_params, lr=lr_p1, weight_decay=1e-2)
    scaler = torch.cuda.amp.GradScaler()
    
    for epoch in range(epochs_p1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Phase 1 - Epoch {epoch+1}/{epochs_p1} [Train]")
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer_p1.zero_grad()
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer_p1)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        # Validation
        model.eval()
        val_probs, val_targets = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device)
                with torch.cuda.amp.autocast():
                    p1 = torch.sigmoid(model(imgs))
                    p2 = torch.sigmoid(model(torch.flip(imgs, dims=[-1])))
                    probs = (p1 + p2) / 2.0
                val_probs.extend(probs.cpu().numpy())
                val_targets.extend(labels.numpy())
        val_auc = roc_auc_score(val_targets, val_probs)
        val_acc = np.mean((np.array(val_probs) > 0.5) == np.array(val_targets))
        print(f"Phase 1 - Epoch {epoch+1}/{epochs_p1} Complete | Val Acc: {val_acc*100:.2f}% | Val AUC: {val_auc:.4f}\n")
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_weights = model.state_dict()

    # Phase 2: Fine-tuning all layers
    print("Phase 2: Fine-tuning all layers")
    for p in unwrapped.spatial_backbone.parameters():
        p.requires_grad = True
        
    backbone_params = [p for p in unwrapped.spatial_backbone.parameters()]
    other_params = [p for n, p in model.named_parameters() if "spatial_backbone" not in n]
    
    optimizer_p2 = torch.optim.AdamW([
        {'params': backbone_params, 'lr': lr_backbone},
        {'params': other_params, 'lr': lr_head}
    ], weight_decay=1e-2)
    scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=epochs_p2)
    
    for epoch in range(epochs_p2):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Phase 2 - Epoch {epoch+1}/{epochs_p2} [Train]")
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer_p2.zero_grad()
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer_p2)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        scheduler_p2.step()
        
        # Validation
        model.eval()
        val_probs, val_targets = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device)
                with torch.cuda.amp.autocast():
                    p1 = torch.sigmoid(model(imgs))
                    p2 = torch.sigmoid(model(torch.flip(imgs, dims=[-1])))
                    probs = (p1 + p2) / 2.0
                val_probs.extend(probs.cpu().numpy())
                val_targets.extend(labels.numpy())
        val_auc = roc_auc_score(val_targets, val_probs)
        val_acc = np.mean((np.array(val_probs) > 0.5) == np.array(val_targets))
        print(f"Phase 2 - Epoch {epoch+1}/{epochs_p2} Complete | Val Acc: {val_acc*100:.2f}% | Val AUC: {val_auc:.4f}\n")
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_weights = model.state_dict()
            
    if best_weights is not None:
        model.load_state_dict(best_weights)
        
    # Youden's J Optimal Decision Threshold Calibration on Validation Set
    model.eval()
    val_probs, val_targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            with torch.cuda.amp.autocast():
                p1 = torch.sigmoid(model(imgs))
                p2 = torch.sigmoid(model(torch.flip(imgs, dims=[-1])))
                probs = (p1 + p2) / 2.0
            val_probs.extend(probs.cpu().numpy())
            val_targets.extend(labels.numpy())
            
    from sklearn.metrics import roc_curve
    fpr, tpr, thresholds = roc_curve(val_targets, val_probs)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    optimal_thresh = float(thresholds[best_idx]) if len(thresholds) > best_idx else 0.5
    print(f"Validation Calibration Complete | Optimal Decision Threshold (Youden's J): {optimal_thresh:.4f}")
    
    # Save model checkpoint with metadata
    checkpoint = {
        'state_dict': model.state_dict(),
        'optimal_threshold': optimal_thresh,
        'best_val_auc': best_val_auc
    }
    torch.save(checkpoint, '/kaggle/working/deepfake_convnext_v2.pth')
    return model, optimal_thresh

model_dual = build_model(use_fft=True)
model_dual, opt_thresh_dual = train_two_phase(model_dual, train_loader, val_loader)

model_dual.eval()
test_probs, test_targets = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Evaluating Test Set"):
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast():
            p1 = torch.sigmoid(model_dual(imgs))
            p2 = torch.sigmoid(model_dual(torch.flip(imgs, dims=[-1])))
            probs = (p1 + p2) / 2.0
        test_probs.extend(probs.cpu().numpy())
        test_targets.extend(labels.numpy())
        
test_preds = (np.array(test_probs) > opt_thresh_dual).astype(int)
print(f"\nTest Evaluation Results (Optimal Threshold = {opt_thresh_dual:.4f}):")
print(classification_report(test_targets, test_preds, target_names=["Real", "Fake"]))
print(f"Test AUC: {roc_auc_score(test_targets, test_probs):.4f}")

## 7. Frequency Stream Ablation Study

In [ ]:
# Ablation study comparing Spatial-Only vs Dual-Stream model
print("Ablation Study: Spatial-Only vs Dual-Stream")
model_spatial = build_model(use_fft=False)
model_spatial, opt_thresh_spatial = train_two_phase(model_spatial, train_loader, val_loader)

model_spatial.eval()
spatial_probs = []
with torch.no_grad():
    for imgs, _ in test_loader:
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast():
            p1 = torch.sigmoid(model_spatial(imgs))
            p2 = torch.sigmoid(model_spatial(torch.flip(imgs, dims=[-1])))
            probs = (p1 + p2) / 2.0
        spatial_probs.extend(probs.cpu().numpy())
        
spatial_auc = roc_auc_score(test_targets, spatial_probs)
spatial_acc = np.mean((np.array(spatial_probs) > opt_thresh_spatial) == np.array(test_targets))
dual_auc = roc_auc_score(test_targets, test_probs)
dual_acc = np.mean((np.array(test_probs) > opt_thresh_dual) == np.array(test_targets))

print("\nAblation Results:")
print(f"Spatial-Only (ConvNeXt):      Acc = {spatial_acc*100:.2f}% | AUC = {spatial_auc:.4f}")
print(f"Dual-Stream (ConvNeXt + FFT): Acc = {dual_acc*100:.2f}% | AUC = {dual_auc:.4f}")
print(f"Delta (FFT Improvement):      Acc = +{(dual_acc - spatial_acc)*100:.2f}% | AUC = +{dual_auc - spatial_auc:.4f}")

## 8. Leave-One-Type-Out (LOTO) Benchmark

In [ ]:
# Evaluate generalization on held-out manipulation type
print(f"LOTO Benchmark: Holding out '{HELD_OUT_TYPE}'")
loto_train_samples = [s for s in train_samples if HELD_OUT_TYPE.lower() not in s[0].lower()]
loto_val_samples = [s for s in val_samples if HELD_OUT_TYPE.lower() not in s[0].lower()]
loto_test_samples = [s for s in test_samples if s[1] == 0 or HELD_OUT_TYPE.lower() in s[0].lower()]

print(f"LOTO Train: {len(loto_train_samples)} | Val: {len(loto_val_samples)} | Test: {len(loto_test_samples)}")

loto_train_labels = [s[1] for s in loto_train_samples]
loto_counts = np.bincount(loto_train_labels)
loto_weights = 1. / loto_counts
loto_sample_weights = [loto_weights[l] for l in loto_train_labels]
loto_sampler = WeightedRandomSampler(weights=loto_sample_weights, num_samples=len(loto_sample_weights), replacement=True)

loto_train_loader = DataLoader(DeepfakeDataset(loto_train_samples, train_transform), batch_size=batch_sz, sampler=loto_sampler, num_workers=2)
loto_val_loader = DataLoader(DeepfakeDataset(loto_val_samples, eval_transform), batch_size=batch_sz, shuffle=False, num_workers=2)
loto_test_loader = DataLoader(DeepfakeDataset(loto_test_samples, eval_transform), batch_size=batch_sz, shuffle=False, num_workers=2)

model_loto = build_model(use_fft=True)
model_loto, opt_thresh_loto = train_two_phase(model_loto, loto_train_loader, loto_val_loader)

model_loto.eval()
loto_probs, loto_targets = [], []
with torch.no_grad():
    for imgs, labels in loto_test_loader:
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast():
            p1 = torch.sigmoid(model_loto(imgs))
            p2 = torch.sigmoid(model_loto(torch.flip(imgs, dims=[-1])))
            probs = (p1 + p2) / 2.0
        loto_probs.extend(probs.cpu().numpy())
        loto_targets.extend(labels.numpy())
        
loto_auc = roc_auc_score(loto_targets, loto_probs)
loto_acc = np.mean((np.array(loto_probs) > opt_thresh_loto) == np.array(loto_targets))
print(f"Unseen Manipulation '{HELD_OUT_TYPE}' Accuracy: {loto_acc*100:.2f}% | AUC: {loto_auc:.4f}")

## 9. Model Export

In [ ]:
# Save model checkpoint and export to ONNX format
export_pth_path = "/kaggle/working/deepfake_convnext_v2.pth"
unwrapped_model = model_dual.module if hasattr(model_dual, 'module') else model_dual
checkpoint = {
    'state_dict': unwrapped_model.state_dict(),
    'optimal_threshold': opt_thresh_dual
}
torch.save(checkpoint, export_pth_path)
print(f"Saved PyTorch checkpoint with optimal_threshold ({opt_thresh_dual:.4f}): {export_pth_path}")

try:
    import onnx
    export_onnx_path = "/kaggle/working/deepfake_convnext_v2.onnx"
    unwrapped_model.eval()
    dummy_in = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    torch.onnx.export(
        unwrapped_model,
        dummy_in,
        export_onnx_path,
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )
    print(f"Exported ONNX model: {export_onnx_path}")
except Exception as e:
    print(f"ONNX export failed: {e}")